In [ ]:
import weather
import date_time
import read_images
import llm_calls
import asyncio
import vector_db as db
import doc_parsing as dp
import csv

In [2]:
zip_code = weather.get_zip_code()
current_weather = weather.get_weather_info(zip_code)
current_date_time = date_time.get_date_time()


print(f'The current weather for zip code {zip_code} at {current_date_time} is:\n{current_weather}')
print('\nPlease select an image that depicts a plausible situation given the current weather conditions:\n')
image = read_images.read_image_choice()

The current weather for zip code 22193 at 17:04 on March 18, 2026 is:
{'short_forecast': 'Sunny', 'temperature': '42 °F', 'probability_of_precipitation': 1, 'relative_humidity': 29, 'wind_speed': '6 mph'}

Please select an image that depicts a plausible situation given the current weather conditions:

1: blind_right_daytime.png
2: blind_right_snow.png
3: highway_left_night.png
4: highway_left_rain.png


In [ ]:
tasks = [
    llm_calls.describe_conditions(),
    llm_calls.describe_scene(image)
]

inputs = await asyncio.gather(*tasks)
conditions = inputs[0]
scene = inputs[1]

inputs

In [ ]:
db.update_vector_db()
# db.client.delete_collection('motorcycle_cornering_docs')

context = db.query_vector_db(conditions=conditions, scene=scene)
context

Vector database up-to-date.


In [ ]:
rec = llm_calls.recommend_approach(inputs[0], inputs[1], context)
rec

In [22]:
new_row = {
    'traction': conditions.traction,
    'visibility': conditions.visibility,
    'traffic': conditions.traffic,
    'curve_sharpness': scene.curve_sharpness,
    'curve_visibility': scene.visible_curve_length,
    'oncoming_traffic': scene.oncoming_traffic,
    'entry_speed': rec.entry_speed,
    'braking': rec.braking,
    'lean_angle': rec.lean_angle,
    'lane_position': rec.lane_position
}

with open('recommendations.csv', 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=new_row.keys())
    writer.writerow(new_row)

print("Row added to recommendations.csv")

Row added to recommendations.csv
